# 跨云平台成果导出 / 导入工具

用于在云平台 A 中止后，把成果打包带走，在新平台 B 上继续工作。
按需选择下面不同的 Cell，**互不依赖**：

| Cell | 用途 | 适用场景 |
|------|------|----------|
| **Cell 1** | 导出训练成果（checkpoint + fonts + charsets） | 跨平台**继续训练** |
| **Cell 2** | 导出推理 PNG（`samples_*/inference/gen/`） | 跨平台**做 SVG 转换** |
| **Cell 3** | 导入训练成果 | 新平台接着训练 |
| **Cell 4** | 导入推理 PNG | 新平台（CPU）做 SVG 转换 |

## 重要前提
- 新旧平台 `fonts/` 下的目标字体**文件名必须一致**（checkpoint 路径与 charset 子目录都按字体名派生）。
- 继续训练的新平台仍需跑「数据准备 Cell」重建 `data/`。
- 代码本身（本项目）通过 git clone / 上传获取，**本 notebook 不打包代码**。

In [ ]:
# ============================================================
# Cell 1【导出训练成果 / 平台 A】：打包 checkpoint + fonts + charsets
# ============================================================
# 适用于：跨平台「继续训练」。
# 无需打包：data/（可重建）、推理 PNG（用 Cell 2 单独打包）、项目代码（git 获取）

import os
import zipfile
from pathlib import Path


# ============ 自动定位项目根目录 ============
def _find_project_root():
    if os.path.isdir("scripts") and os.path.exists("requirements.txt"):
        return Path.cwd()
    for search_root in [os.getcwd(), "/workspace", "/home", "/"]:
        if not os.path.isdir(search_root):
            continue
        for entry in sorted(os.listdir(search_root)):
            cand = os.path.join(search_root, entry)
            if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "scripts")):
                return Path(cand)
    return None

_proj = _find_project_root()
if _proj is None:
    raise SystemExit("未找到项目根目录（含 scripts/），请手动设置 PROJECT_ROOT 后重跑。")
os.chdir(_proj)
print(f"已切换到项目根目录: {os.getcwd()}")
# ===========================================

# ---- 可调整 ----
DIRS_TO_PACK = ["checkpoints", "fonts", "charsets"]
OUTPUT_ZIP_NAME = "hanzi_train_transfer.zip"

def _pick_output_dir():
    for base in [Path("/workspace"), Path.cwd()]:
        if base.exists() and os.access(base, os.W_OK):
            return base
    return Path.cwd()

OUTPUT_ZIP = _pick_output_dir() / OUTPUT_ZIP_NAME
# -----------------

missing = [d for d in DIRS_TO_PACK if not Path(d).exists()]
if missing:
    print("以下目录不存在，将被跳过：", missing)

present = [d for d in DIRS_TO_PACK if Path(d).exists()]

ckpt_dir = Path("checkpoints")
ckpt_files = list(ckpt_dir.glob("*.pth")) if ckpt_dir.exists() else []
if ckpt_dir in map(Path, present) and not ckpt_files:
    print("checkpoints/ 目录存在但没有 .pth 文件，尚未产生任何 checkpoint！")

if not present:
    raise SystemExit("没有任何可打包的训练成果目录，导出终止。")

if Path(OUTPUT_ZIP).exists():
    Path(OUTPUT_ZIP).unlink()

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for d in present:
        for root, _, files in os.walk(d):
            for f in files:
                fp = Path(root) / f
                zf.write(fp, fp)

size_mb = Path(OUTPUT_ZIP).stat().st_size / 1024 / 1024
print(f"已导出训练成果: {Path(OUTPUT_ZIP).resolve()}  ({size_mb:.1f} MB)")
print("   包含目录:", present)
print(f"   checkpoint 文件数: {len(ckpt_files)}")
print("\n请将此 zip 下载到本地，上传到新平台后运行 Cell 3。")

In [ ]:
# ============================================================
# Cell 2【导出推理 PNG / 平台 A】：打包 samples_*/inference/gen/ 的 PNG
# ============================================================
# 适用于：跨平台「做 SVG 转换」（CPU 平台用，无需 checkpoint）。
# 前提：已在本平台跑完推理（Cell 5），samples_*/inference/gen/ 里有 PNG。

import os
import glob
import zipfile
from pathlib import Path


# ============ 自动定位项目根目录 ============
def _find_project_root():
    if os.path.isdir("scripts") and os.path.exists("requirements.txt"):
        return Path.cwd()
    for search_root in [os.getcwd(), "/workspace", "/home", "/"]:
        if not os.path.isdir(search_root):
            continue
        for entry in sorted(os.listdir(search_root)):
            cand = os.path.join(search_root, entry)
            if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "scripts")):
                return Path(cand)
    return None

_proj = _find_project_root()
if _proj is None:
    raise SystemExit("未找到项目根目录（含 scripts/），请手动设置 PROJECT_ROOT 后重跑。")
os.chdir(_proj)
print(f"已切换到项目根目录: {os.getcwd()}")
# ===========================================

# ---- 可调整 ----
INFER_GEN_GLOBS = ["samples_*/inference/gen"]
OUTPUT_ZIP_NAME = "hanzi_infer_transfer.zip"

def _pick_output_dir():
    for base in [Path("/workspace"), Path.cwd()]:
        if base.exists() and os.access(base, os.W_OK):
            return base
    return Path.cwd()

OUTPUT_ZIP = _pick_output_dir() / OUTPUT_ZIP_NAME
# -----------------

infer_gen_dirs = []
for g in INFER_GEN_GLOBS:
    for d in glob.glob(g):
        if os.path.isdir(d):
            infer_gen_dirs.append(d)
infer_gen_dirs = sorted(set(infer_gen_dirs))

if not infer_gen_dirs:
    raise SystemExit(
        "未检测到 samples_*/inference/gen/ 目录。\n"
        "请先在本平台跑完推理（Cell 5）生成 PNG，再运行本 Cell。"
    )

print(f"检测到推理产出目录: {infer_gen_dirs}")

if Path(OUTPUT_ZIP).exists():
    Path(OUTPUT_ZIP).unlink()

png_count = 0
with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for d in infer_gen_dirs:
        for root, _, files in os.walk(d):
            for f in files:
                if f.endswith(".png"):
                    fp = Path(root) / f
                    zf.write(fp, fp)  # 保留 samples_*/inference/gen 相对结构
                    png_count += 1

size_mb = Path(OUTPUT_ZIP).stat().st_size / 1024 / 1024
print(f"已导出推理 PNG: {Path(OUTPUT_ZIP).resolve()}  ({size_mb:.1f} MB)")
print(f"   共 {png_count} 张 PNG")
print("\n请将此 zip 下载到本地，上传到 CPU 平台后运行 Cell 4。")

In [ ]:
# ============================================================
# Cell 3【导入训练成果 / 平台 B】：解压 checkpoint + fonts + charsets
# ============================================================
# 注意：解压前请确认新平台已通过 git 拿到本项目代码框架，
# 且 fonts/ 下的目标字体文件名与平台 A 完全一致。

import os
from pathlib import Path
import zipfile

# ============ 自动定位项目根目录 ============
def _find_project_root():
    if os.path.isdir("scripts") and os.path.exists("requirements.txt"):
        return Path.cwd()
    for search_root in [os.getcwd(), "/workspace", "/home", "/"]:
        if not os.path.isdir(search_root):
            continue
        for entry in sorted(os.listdir(search_root)):
            cand = os.path.join(search_root, entry)
            if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "scripts")):
                return Path(cand)
    return None

_proj = _find_project_root()
if _proj is None:
    raise SystemExit("未找到项目根目录（含 scripts/），请先在新平台克隆项目代码。")
os.chdir(_proj)
print(f"已切换到项目根目录: {os.getcwd()}")
# ===========================================

# ---- 可调整：zip 文件名与解压根目录 ----
ZIP_NAME = "hanzi_train_transfer.zip"   # 上传后的 zip 文件名
EXTRACT_ROOT = "."                       # 解压到项目根目录（目录结构会自动还原）
# -------------------------------------

def _locate_zip(name):
    if os.path.isabs(name):
        p = Path(name)
        return p if p.exists() else None
    for base in [Path.cwd(), Path("/workspace"), Path.home(), Path("/")]:
        p = base / name
        if p.exists():
            return p
    return None

ZIP_PATH = _locate_zip(ZIP_NAME)
if ZIP_PATH is None:
    raise SystemExit(
        f"找不到 {ZIP_NAME}。已搜索: 当前目录、/workspace、$HOME、/。\n"
        "请将 zip 上传到 /workspace（或 notebook 所在目录）后重跑本 Cell。"
    )
print(f"使用 zip: {ZIP_PATH}")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = zf.namelist()
    zf.extractall(EXTRACT_ROOT)

print(f"已解压 {len(names)} 个文件到 {EXTRACT_ROOT}")
top_dirs = sorted({n.split('/')[0] for n in names if '/' in n})
print("   还原的顶层目录:", top_dirs)

# 校验关键点
ckpt = list(Path("checkpoints").glob("*.pth")) if Path("checkpoints").exists() else []
print(f"\ncheckpoints/*.pth 数量: {len(ckpt)}")
for p in ckpt:
    print("   -", p.name)

print("\n下一步：跑「数据准备 Cell」重建 data/（字体名需一致），再运行训练 Cell；")
print("训练 Cell 会自动检测到 checkpoints 并 resume 继续训练。")


In [ ]:
# ============================================================
# Cell 4【导入推理 PNG / 平台 B】：解压 samples_*/inference/gen/ 的 PNG
# ============================================================
# 适用于：CPU 平台做 SVG 转换。解压后直接运行 SVG 转换 Cell 即可，无需 GPU。

import os
import glob
from pathlib import Path
import zipfile

# ============ 自动定位项目根目录 ============
def _find_project_root():
    if os.path.isdir("scripts") and os.path.exists("requirements.txt"):
        return Path.cwd()
    for search_root in [os.getcwd(), "/workspace", "/home", "/"]:
        if not os.path.isdir(search_root):
            continue
        for entry in sorted(os.listdir(search_root)):
            cand = os.path.join(search_root, entry)
            if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "scripts")):
                return Path(cand)
    return None

_proj = _find_project_root()
if _proj is None:
    raise SystemExit("未找到项目根目录（含 scripts/），请先在新平台克隆项目代码。")
os.chdir(_proj)
print(f"已切换到项目根目录: {os.getcwd()}")
# ===========================================

# ---- 可调整：zip 文件名与解压根目录 ----
ZIP_NAME = "hanzi_infer_transfer.zip"   # 上传后的 zip 文件名
EXTRACT_ROOT = "."                       # 解压到项目根目录
# -------------------------------------

def _locate_zip(name):
    if os.path.isabs(name):
        p = Path(name)
        return p if p.exists() else None
    for base in [Path.cwd(), Path("/workspace"), Path.home(), Path("/")]:
        p = base / name
        if p.exists():
            return p
    return None

ZIP_PATH = _locate_zip(ZIP_NAME)
if ZIP_PATH is None:
    raise SystemExit(
        f"找不到 {ZIP_NAME}。已搜索: 当前目录、/workspace、$HOME、/。\n"
        "请将 zip 上传到 /workspace（或 notebook 所在目录）后重跑本 Cell。"
    )
print(f"使用 zip: {ZIP_PATH}")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = zf.namelist()
    zf.extractall(EXTRACT_ROOT)

print(f"已解压 {len(names)} 个文件到 {EXTRACT_ROOT}")

# 校验推理产出 PNG 是否已还原
gen_dirs = sorted(glob.glob("samples_*/inference/gen"))
if gen_dirs:
    print(f"检测到推理产出目录: {gen_dirs}")
    for g in gen_dirs:
        n = len(glob.glob(os.path.join(g, "*.png")))
        print(f"   {g}: {n} 张 PNG")
    print("\n下一步：直接运行 SVG 转换 Cell（无需 GPU），把 PNG 转成 SVG。")
else:
    print("未检测到推理产出目录，请确认 zip 内容是否正确。")
